In [1]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langsmith import Client
from langchain.tools import tool
import requests

In [2]:
from langchain.agents import create_react_agent, AgentExecutor

In [3]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [4]:
search_tool = TavilySearchResults(max_results=2)

C:\Users\Mohamad Badrawy\AppData\Local\Temp\ipykernel_6308\919418145.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=2)


In [5]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [6]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'title': 'The latest AI news we announced in September 2026',
  'url': 'https://blog.google/innovation-and-ai/technology/ai/google-ai-updates-september-2026',
  'content': "Speed 0.75X 1X 1.5X 2X\n\n Read AI-generated summary \n\n   Check out the latest AI news from September 2026 in our monthly roundup.\n   Gemini 4 Argon leads our new frontier models with advanced reasoning for complex tasks.\n   We’ve launched natural voice models and new app integrations to simplify your daily workflow.\n   Exciting scientific breakthroughs include mapping human DNA and tracking global methane from space.\n   New tools like Googlebook and Gemini for Windows help you get more done effortlessly.\n\n Summaries were generated by Google AI. Generative AI is experimental. [...] September's headline belongs to Gemini 4 Argon, our newest frontier model. Built with advanced reasoning and a 1-million-token output limit, it’s designed to tackle complex challenges — especially in cybersecurity defense. The r

In [7]:
# ==========================================
# LLM
# ==========================================

llm = ChatGroq(model="qwen/qwen3.8-27b")

In [8]:
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content='Why did the AI go to therapy?\n\nBecause it had too many *neural* issues! 🧠😂', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 18, 'total_tokens': 44, 'completion_time': 0.054768687, 'completion_tokens_details': None, 'prompt_time': 0.001150302, 'prompt_tokens_details': None, 'queue_time': 0.016584088, 'total_time': 0.055918989}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_8999620a6e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a18e20-e483-70d0-98e1-1d7b1ae7ae51-0', usage_metadata={'input_tokens': 18, 'output_tokens': 26, 'total_tokens': 44})

In [9]:
# ==========================================
# PROMPT
# ==========================================

client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

In [10]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [11]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

In [13]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [14]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [15]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": (
        "Find the capital of Egypt"
        "and then find its current weather."
    )
})




> Entering new AgentExecutor chain...
Question: Find the capital of Egypt and then find its current weather.
Thought: First, I need to identify the capital of Egypt. I can use the search tool to confirm this fact.
Action: tavily_search_results_json
Action Input: capital of Egypt[{'title': 'What Is The Capital City Of Egypt?', 'url': 'https://www.worldatlas.com/articles/what-is-the-capital-city-of-egypt.html', 'content': "## What Is The Capital Of Egypt And Where Is It Located?\n\nCairo is Egypt’s capital city and also the most populous city in the country. The city is located in Lower Egypt which is a section of Egypt encompassing the fertile Nile delta and the land between Upper Egypt and the northern Mediterranean coast. Cairo is at a distance of 165 km to the south of the Mediterranean coast. It lies 120 km to the west of the Suez Canal. The city is based along the banks of the world’s longest river, the Nile River. More specifically, Cairo is based immediately south of the point 

KeyboardInterrupt: 

In [16]:
print(response["output"])

TypeError: 'AIMessage' object is not subscriptable